# 06 HumAID: a usefulness score and a threshold for the priority list

Member 5. The priority list ranks tweets so that the most useful ones come first. The survey we follow says usefulness estimation needs a predefined threshold. This notebook defines the score, sets the threshold on the **validation** split only, applies it once to the **test** split, and writes a small demo list.

**This is a proxy, and the report must say so.** HumAID has no usefulness labels. A tweet counts as "useful" here when its human label is one of the six actionable classes (`ACTIONABLE`, the list agreed in notebook 02). The score is the classifier's total probability on those classes. Everything below measures how well that score recovers the actionable label. It does not measure whether a responder would find the tweet useful.

| step | data | what happens |
| --- | --- | --- |
| score | validation and test | probability mass on the actionable classes, from the notebook 02 model |
| threshold | validation | pick the cut-off that maximises F-beta for the proxy label |
| check | test | apply the cut-off once; report precision, recall and precision at the top k |
| stability | test, by event | does the cut-off carry over to each event? (needs `events.parquet` from notebook 05) |

**Needs the model saved by notebook 02** (`humaid/transformer` on Drive) and a GPU or CPU runtime (a GPU takes under a minute; a CPU takes a few minutes). Clear outputs before committing. Never commit the model or the data.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "scikit-learn", "pyarrow"], check=True)

import os
import numpy as np
import pandas as pd
import torch
from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
DATA = f"{ROOT}/humaid"
MODEL_DIR = f"{DATA}/transformer"             # saved by notebook 02
EVENTS_PARQUET = f"{DATA}/events.parquet"     # saved by notebook 05 (optional, for the per-event check)
SEED = 42
BETA = 1.0                 # 1.0 balances precision and recall; 2.0 favours recall (miss fewer urgent tweets, review more noise)
KS = (10, 20, 50, 100)     # list lengths for precision at the top k

ACTIONABLE = ["infrastructure_and_utility_damage", "injured_or_dead_people", "displaced_people_and_evacuations",
              "requests_or_urgent_needs", "missing_or_found_people", "caution_and_advice"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## Load the model and the two splits

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device).eval()
LABELS = [model.config.id2label[i] for i in range(model.config.num_labels)]
missing = [c for c in ACTIONABLE if c not in LABELS]
assert not missing, f"These actionable classes are not in the model's labels: {missing}"

val = pd.read_parquet(f"{DATA}/validation.parquet")
test = pd.read_parquet(f"{DATA}/test.parquet")
print(len(val), "validation tweets,", len(test), "test tweets,", len(LABELS), "classes")

## Score

The usefulness score of a tweet is the sum of the model's probabilities over the actionable classes. It lies between 0 and 1 and does not depend on which single class wins. The proxy label is 1 when the human label is actionable.

In [ ]:
def score_probs(texts, model, tokenizer, device, batch=128, max_len=128):
    """Class probabilities for each text, in batches. Rows follow the order of texts."""
    texts, out = list(texts), []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch):
            enc = tokenizer(texts[i:i + batch], truncation=True, max_length=max_len, padding=True, return_tensors="pt").to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
                logits = model(**enc).logits
            out.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.vstack(out)


def usefulness_scores(probs, labels, actionable):
    """Probability mass on the actionable classes."""
    idx = [labels.index(c) for c in actionable]
    return probs[:, idx].sum(axis=1)


def proxy_labels(label_series, actionable):
    """1 when the human label is an actionable class, else 0."""
    return label_series.isin(actionable).to_numpy().astype(int)


probs_val = score_probs(val["text"], model, tokenizer, device)
probs_test = score_probs(test["text"], model, tokenizer, device)
s_val, s_test = usefulness_scores(probs_val, LABELS, ACTIONABLE), usefulness_scores(probs_test, LABELS, ACTIONABLE)
y_val, y_test = proxy_labels(val["label"], ACTIONABLE), proxy_labels(test["label"], ACTIONABLE)
print("actionable share: validation", round(float(y_val.mean()), 3), "| test", round(float(y_test.mean()), 3))

## Threshold on the validation split

For each candidate cut-off the table gives precision, recall, F1 and the share of tweets flagged. The cut-off with the best F-beta on validation is the one used from here on. The test split is not touched until the next section.

In [ ]:
def prf(flagged, y):
    """Precision, recall, F1 and flagged share for a boolean flag array against 0/1 labels."""
    tp = int((flagged & (y == 1)).sum())
    p = tp / max(int(flagged.sum()), 1)
    r = tp / max(int((y == 1).sum()), 1)
    return {"precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0.0, "flagged_share": float(flagged.mean())}


def threshold_table(scores, y, thresholds=None):
    """Metrics at each candidate threshold."""
    thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2) if thresholds is None else thresholds
    return pd.DataFrame([{"threshold": float(t), **prf(scores >= t, y)} for t in thresholds])


def fbeta_threshold(table, beta=BETA):
    """Threshold with the highest F-beta, and the table with an fbeta column."""
    b2 = beta ** 2
    denom = (b2 * table["precision"] + table["recall"]).replace(0, np.nan)
    table = table.assign(fbeta=((1 + b2) * table["precision"] * table["recall"] / denom).fillna(0.0))
    return float(table.loc[table["fbeta"].idxmax(), "threshold"]), table


table_val = threshold_table(s_val, y_val)
THETA, table_val = fbeta_threshold(table_val, BETA)
print("threshold chosen on validation:", THETA, "(beta =", BETA, ")")
table_val.round(3)

In [ ]:
import matplotlib.pyplot as plt


def plot_threshold_curve(table, theta, title):
    """Precision, recall and F1 against the threshold, with the chosen threshold marked."""
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for col in ("precision", "recall", "f1"):
        ax.plot(table["threshold"], table[col], marker="o", ms=3, label=col)
    ax.axvline(theta, color="0.4", ls="--", label=f"chosen {theta}")
    ax.set_xlabel("usefulness threshold")
    ax.set_ylim(0, 1.02)
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    return fig


plot_threshold_curve(table_val, THETA, "Proxy usefulness on the validation split");

## Apply once to the test split

Three views of the test result: the metrics at the chosen threshold, the same metrics for the simplest alternative (flag a tweet when its single most likely class is actionable, no threshold), and the precision of the top k of the ranked list. If the threshold does not beat the simple rule, say so in the report.

In [ ]:
def apply_once(scores, y, theta):
    """Metrics at the chosen threshold, with the base rate of actionable tweets."""
    return {"threshold": theta, "n": len(y), "base_rate": round(float(y.mean()), 3),
            **{k: round(v, 3) for k, v in prf(scores >= theta, y).items()}}


def argmax_rule(probs, labels, y, actionable):
    """Flag a tweet when its single most likely class is actionable (no threshold)."""
    flagged = np.isin(np.array(labels)[probs.argmax(axis=1)], actionable)
    return {k: round(v, 3) for k, v in prf(flagged, y).items()}


def precision_at_k(scores, y, ks):
    """Share of actionable tweets among the k highest-scoring ones."""
    order = np.argsort(-scores, kind="stable")
    return {k: round(float(y[order[:k]].mean()), 3) for k in ks if k <= len(y)}


result = pd.DataFrame({"chosen threshold": apply_once(s_test, y_test, THETA),
                       "argmax rule": {"threshold": "none", "n": len(y_test), "base_rate": round(float(y_test.mean()), 3),
                                       **argmax_rule(probs_test, LABELS, y_test, ACTIONABLE)}})
print(result)
print("precision at top k (test):", precision_at_k(s_test, y_test, KS))

### Is the score calibrated?

A score of 0.8 should mean that about 80% of such tweets are actionable. If the bars sit far from the diagonal, the threshold still works as a cut-off but the score must not be read as a probability in the report or the demo.

In [ ]:
def reliability_table(scores, y, n_bins=10):
    """Mean score and actual actionable rate in equal-width score bins."""
    bins = pd.cut(scores, np.linspace(0, 1, n_bins + 1), include_lowest=True)
    t = pd.DataFrame({"bin": bins, "score": scores, "y": y}).groupby("bin", observed=True).agg(
        n=("y", "size"), mean_score=("score", "mean"), actual_rate=("y", "mean"))
    return t.reset_index(drop=True)


def plot_reliability(rel, title):
    """Actual actionable rate against mean score per bin, with the diagonal for a calibrated score."""
    fig, ax = plt.subplots(figsize=(4.8, 4.5))
    ax.plot([0, 1], [0, 1], color="0.6", ls="--")
    ax.plot(rel["mean_score"], rel["actual_rate"], marker="o")
    ax.set_xlabel("mean usefulness score")
    ax.set_ylabel("share actually actionable")
    ax.set_title(title)
    fig.tight_layout()
    return fig


rel = reliability_table(s_test, y_test)
print(rel.round(3))
plot_reliability(rel, "Calibration on the test split");

## Does the threshold carry over to each event?

The validation split mixes all 19 events, so a threshold chosen on it inherits the same optimism as the 0.750 macro-F1 (see notebook 05). This cell applies the same threshold to the test tweets of each event. If precision or recall swing widely, say that the threshold should be re-checked on the first labelled tweets of a new event. It runs only when notebook 05 has saved `events.parquet`.

In [ ]:
def per_event_at_threshold(df, scores, y, theta, events):
    """Precision, recall and flagged share of the chosen threshold for each event (matched on exact text)."""
    lookup = events.drop_duplicates("text").set_index("text")["event"]
    ev = df["text"].map(lookup).to_numpy()
    rows = []
    for e in sorted({x for x in ev if isinstance(x, str)}):
        m = ev == e
        rows.append({"event": e, "n": int(m.sum()), "base_rate": round(float(y[m].mean()), 3),
                     **{k: round(v, 3) for k, v in prf(scores[m] >= theta, y[m]).items()}})
    return pd.DataFrame(rows).sort_values("n", ascending=False).reset_index(drop=True)


if os.path.exists(EVENTS_PARQUET):
    by_event = per_event_at_threshold(test, s_test, y_test, THETA, pd.read_parquet(EVENTS_PARQUET))
    print(by_event.to_string(index=False))
    print("precision range:", by_event["precision"].min(), "to", by_event["precision"].max(),
          "| recall range:", by_event["recall"].min(), "to", by_event["recall"].max())
else:
    by_event = None
    print("events.parquet not found: run notebook 05 first for this check.")

## Demo priority list

The top tweets above the threshold, with the class the model thinks they belong to. Saved as a small CSV for the demo. The score ranks tweets; it does not say that a tweet is true or recent.

In [ ]:
def priority_list(df, probs, scores, labels, theta, k=20):
    """The k highest-scoring tweets at or above the threshold, with the most likely class."""
    order = [i for i in np.argsort(-scores, kind="stable") if scores[i] >= theta][:k]
    return pd.DataFrame({"rank": range(1, len(order) + 1), "score": scores[order].round(3),
                         "predicted_class": np.array(labels)[probs[order].argmax(axis=1)],
                         "text": df["text"].to_numpy()[order]})


demo = priority_list(test, probs_test, s_test, LABELS, THETA, 20)
demo.to_csv(f"{DATA}/priority_demo.csv", index=False)
demo

## Paste back

Run this cell and paste its output, so the numbers can go into the report and the demo.

In [ ]:
def paste_back(theta, beta, table_val, result, p_at_k, rel, by_event):
    """One text block: chosen threshold, validation row, test result, precision at k, calibration, per-event spread."""
    row = table_val[table_val["threshold"] == theta].round(3).to_string(index=False)
    lines = [f"threshold {theta} (beta {beta}) chosen on validation", "validation row:", row,
             "test result:", result.to_string(), f"precision at top k (test): {p_at_k}",
             "calibration:", rel.round(3).to_string(index=False)]
    if by_event is not None:
        lines += ["per event:", by_event.to_string(index=False)]
    return "\n".join(lines)


print(paste_back(THETA, BETA, table_val, result, precision_at_k(s_test, y_test, KS), rel, by_event))

## Notes for the report
- Call the actionable classes a **proxy for usefulness**. HumAID has no usefulness labels; say what the six classes are and that the team chose them, and that a tweet can be actionable without being useful (outdated, duplicate, unverifiable) and useful without being actionable.
- State the protocol: the threshold was set on the validation split only, applied once to the test split, with beta as configured. Give the threshold, precision, recall and precision at the top k, and the rule it is compared with.
- Do not read the score as a probability unless the calibration plot supports it.
- The validation and test splits pool all events. Give the per-event spread, or say the check was not run, and tie it to the held-out result in notebook 05.
- The priority list ranks tweets by the text alone. It does not verify them. The satellite evidence and the claim checker are separate steps, and the demo should keep them separate.